# Conjugate gradients for a sparse matrix

Many large linear systems `A x = b` have a matrix `A` that is almost entirely zero. Compressed
sparse row storage keeps only the entries that are not zero, in three arrays. The array `data`
holds the non-zero values row by row. The array `indices` holds the column of each value. The
array `indptr` has one more element than the matrix has rows, and row `i` occupies the positions
from `indptr[i]` up to but not including `indptr[i + 1]` of the other two arrays.

The conjugate gradient method solves the system when `A` is symmetric and positive definite.
Each step needs one product of `A` with a vector, two dot products and three vector updates. The
function below starts from `x = 0` and stops when the norm of the residual `b - A x` is no more
than `rtol` times the norm of `b`, or after `max_iter` steps. The number of steps therefore
depends on the data, and two routines that add up the dot products in a different order may stop
one step apart. The demonstration solves a system of three equations, whose matrix has 2 on the
diagonal and -1 next to it, and prints the solution.

In [ ]:
import numpy as np


def sparse_cg(indptr, indices, data, b, rtol, max_iter):
    """Solve A x = b by conjugate gradients, with A in compressed sparse row form.

    The iteration starts from x = 0. It stops when the norm of the residual b - A x is at most
    `rtol` times the norm of b, or after `max_iter` steps.
    """
    rows = np.repeat(np.arange(len(b)), np.diff(indptr))  # the row of each stored value

    def multiply(v):
        return np.bincount(rows, weights=data * v[indices], minlength=len(b))

    x = np.zeros_like(b)
    r = b.copy()  # the residual b - A x
    p = r.copy()  # the search direction
    rr = r @ r
    limit = rtol * rtol * (b @ b)
    for _ in range(max_iter):
        if rr <= limit:
            break
        q = multiply(p)
        alpha = rr / (p @ q)
        x += alpha * p
        r -= alpha * q
        rr_new = r @ r
        p = r + (rr_new / rr) * p
        rr = rr_new
    return x

In [ ]:
print(sparse_cg(np.array([0, 2, 5, 7], dtype=np.int32),
                np.array([0, 1, 0, 1, 2, 1, 2], dtype=np.int32),
                np.array([2.0, -1.0, -1.0, 2.0, -1.0, -1.0, 2.0]),
                np.array([1.0, 0.0, 1.0]), 1e-10, 100))

The declaration below tells the harness how to call the function and how to check a routine
against it. The three arrays of the matrix have two lengths of their own: `np1`, which is one
more than the number of rows, and `nnz`, the number of values stored. `make_inputs` builds the
matrix of a diffusion problem on a square grid of `size` by `size` points. Every link between
neighbouring points, and every link from a point at the edge to the zero value outside, has a
random conductivity between 0.5 and 2, and a few are set to exactly 0.5, 1 or 2. With a
conductivity of 1 everywhere, the matrix is the five-point Laplacian, which has 4 on the
diagonal and -1 for each neighbour. The right-hand side is random, with a few zeros. For the
smallest grids it is sometimes zero everywhere, and the cap on the steps is sometimes as small
as 1 to 5, so that the routine meets these cases too.

The tolerance is relative to the largest element of the solution. The stopping fraction `rtol`
is 1e-8, and the tolerance is ten times that. One step more or one step fewer changes the
solution by at most 5e-9 of its largest element on grids of side 8 or more, so a correct routine
that stops one step away from the function, because it adds up the dot products in a different
order, stays within the tolerance.

In [ ]:
def make_inputs(size, rng):
    m = size

    def conductivities(shape):
        values = rng.uniform(0.5, 2.0, shape)
        special = rng.random(shape) < 0.05
        values[special] = rng.choice([0.5, 1.0, 2.0], special.sum())
        return values

    horizontal, vertical = conductivities((m, m + 1)), conductivities((m + 1, m))
    up, down, left, right = vertical[:-1], vertical[1:], horizontal[:, :-1], horizontal[:, 1:]
    i, j = np.indices((m, m))
    me = i * m + j
    columns = np.stack([me - m, me - 1, me, me + 1, me + m], axis=-1)
    values = np.stack([-up, -left, up + down + left + right, -right, -down], axis=-1)
    exists = np.stack([i > 0, j > 0, np.ones((m, m), bool), j < m - 1, i < m - 1], axis=-1)
    indptr = np.concatenate(([0], np.cumsum(exists.reshape(-1, 5).sum(axis=1))))
    b = rng.uniform(0.0, 1.0, m * m)
    b[rng.random(m * m) < 0.05] = 0.0
    max_iter = 10_000
    if m < 16:
        if rng.random() < 0.1:
            b[:] = 0.0
        if rng.random() < 0.1:
            max_iter = int(rng.integers(1, 6))
    return {"indptr": indptr.astype(np.int32), "indices": columns[exists].astype(np.int32),
            "data": values[exists], "b": b, "rtol": 1e-8, "max_iter": max_iter}


ROUTINE = {
    "function": sparse_cg,
    "make_inputs": make_inputs,
    "arguments": {"indptr": "int32[np1]", "indices": "int32[nnz]", "data": "float64[nnz]",
                  "b": "float64[n]", "rtol": "float64", "max_iter": "int64"},
    "results": {"x": "float64[n]"},
    "test_sizes": [1, 8, 32, 128],
    "benchmark_size": 320,
    "tolerance": 1e-7,
}